### Set up and load the data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Projects/03_Advanced_house_price _predictor/data/cleaned/house_price_cleaned_data.csv")
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,221900.0,3,1.00,1180.0,5650.0,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650.0
1,6414100192,2014-12-09,538000.0,3,2.25,2570.0,7242.0,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639.0
2,5631500400,2015-02-25,180000.0,2,1.00,770.0,10000.0,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062.0
3,2487200875,2014-12-09,604000.0,4,3.00,1960.0,5000.0,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000.0
4,1954400510,2015-02-18,510000.0,3,2.00,1680.0,8080.0,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503.0


### Create log price

In [2]:
df["log_price"] = np.log1p(df["price"])
df[["price", "log_price"]].head()

,price,log_price
0,221900.0,12.309987
1,538000.0,13.195616
2,180000.0,12.100718
3,604000.0,13.311331
4,510000.0,13.142168


### Convert date into useful features


In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")  
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

df = df.drop(columns=["date"])

### Create house age + renovation features

In [4]:
df["house_age"] = df["sale_year"] - df["yr_built"]
df["is_renovated"] = (df["yr_renovated"] > 0).astype(int)

# If renovated, how many years since renovation (else NaN or 0)
df["years_since_renov"] = np.where(
    df["yr_renovated"] > 0,
    df["sale_year"] - df["yr_renovated"],
    0
)

### Create meaningful ratio

In [5]:
df["basement_ratio"] = df["sqft_basement"] / df["sqft_living15"]
df["above_ratio"] = df["sqft_above"] / df["sqft_living15"]
df["lot_to_living_ratio"] = df["sqft_lot15"] / df["sqft_living15"]

### Handle divide by zero for safety

In [6]:
for c in ["basement_ratio", "above_ratio", "lot_to_living_ratio"]:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

### Fix the categorical variables

In [7]:
df["zipcode"] = df["zipcode"].astype(str)

### Reduce multicollinearity

In [8]:
df = df.drop(columns=["sqft_above"])

### Define X (features) and  y (target)

In [9]:
y = df["log_price"]  # Target 
X = df.drop(columns=["price", "log_price"])  #Features

### Encode categorical features

In [10]:
X = pd.get_dummies(X, columns=["zipcode"], drop_first=True)

### Final Checks

In [11]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())


X shape: (21613, 94)
y shape: (21613,)
Missing values in X: 0


### Save the feature engineered dataset

In [15]:
X.to_csv("C:/Projects/03_Advanced_house_price _predictor/data/cleaned/X_features.csv", index=False)
y.to_csv("C:/Projects/03_Advanced_house_price _predictor/data/cleaned/y_log_price.csv", index=False)
print("Saved engineered features + target.")

Saved engineered features + target.
